In [0]:
# Notebook de ejemplo para DABs
# Este notebook recibe los parámetros catalog y schema del job
# y los usa para leer/escribir en el ambiente correspondiente

# Parámetros del job (inyectados por DABs según el target)
dbutils.widgets.text("catalog", "workspace", "Catálogo de Unity Catalog")
dbutils.widgets.text("schema", "dev", "Esquema de Unity Catalog")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Ejecutando en ambiente: {catalog}.{schema}")

Ejecutando en ambiente: workspace.prb


In [0]:
import random
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType
from datetime import datetime, timedelta

# 1. Listas de datos base para combinar de forma aleatoria
nombres = ["Alejandro", "Sofia", "Mateo", "Valentina", "Santiago", "Camila", "Sebastian", "Isabella", "Nicolas", "Mariana", "Diego", "Gabriela"]
apellidos = ["Rodriguez", "Gomez", "Lopez", "Martinez", "Perez", "Garcia", "Sanchez", "Ramirez", "Torres", "Flores", "Diaz", "Vasquez"]
materias_pool = ["Matematicas", "Programacion I", "Base de Datos", "Calculo Integral", "Fisica Mecanica", "Estadistica", "Estructura de Datos"]
estados = ["Activo", "Inactivo", "Reserva"]

# 2. Función para generar filas de datos dummy
def generar_datos_estudiantes(num_registros=1000):
    datos = []
    fecha_inicio = datetime(2023, 1, 1)
    
    for i in range(num_registros):
        id_estudiante = 20260000 + i
        identificacion = f"ID-{random.randint(10000000, 99999999)}"
        nombre = random.choice(nombres)
        apellido = random.choice(apellidos)
        correo = f"{nombre.lower()}.{apellido.lower()}@universidad.edu"
        
        # Seleccionar entre 2 y 4 materias aleatorias sin repetir
        materias_inscritas = random.sample(materias_pool, random.randint(2, 4))
        # Convertir la lista de materias a un string separado por comas para mantener la estructura simple
        materias_str = ", ".join(materias_inscritas)
        
        promedio = round(random.uniform(1.0, 5.0), 2)  # Escala 1.0 a 5.0
        estado = random.choice(estados)
        fecha_ingreso = (fecha_inicio + timedelta(days=random.randint(0, 1000))).date()
        
        datos.append((id_estudiante, identificacion, nombre, apellido, correo, materias_str, promedio, estado, fecha_ingreso))
    return datos

# 3. Definir el esquema explícito de la tabla
esquema = StructType([
    StructField("id_estudiante", IntegerType(), False),
    StructField("identificacion", StringType(), False),
    StructField("nombre", StringType(), True),
    StructField("apellido", StringType(), True),
    StructField("correo", StringType(), True),
    StructField("materias_inscritas", StringType(), True),
    StructField("promedio_acumulado", FloatType(), True),
    StructField("estado_academico", StringType(), True),
    StructField("fecha_ingreso", DateType(), True)
])

# 4. Crear el DataFrame con los datos generados
datos_dummy = generar_datos_estudiantes(1000)
df_estudiantes = spark.createDataFrame(datos_dummy, schema=esquema)

# 5. Guardar el DataFrame como una tabla Delta en Databricks
# Cambia "default" por el nombre de tu esquema/base de datos si es necesario
nombre_tabla = f"{catalog}.{schema}.estudiantes_inscritos"

df_estudiantes.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(nombre_tabla)

print(f"¡Tabla '{nombre_tabla}' creada exitosamente con {df_estudiantes.count()} registros dummy!")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4801988587202543>, line 60
     53 # 5. Guardar el DataFrame como una tabla Delta en Databricks
     54 # Cambia "default" por el nombre de tu esquema/base de datos si es necesario
     55 nombre_tabla = f"{catalog}.{schema}.estudiantes_inscritos"
     57 df_estudiantes.write \
     58     .mode("overwrite") \
     59     .format("delta") \
---> 60     .saveAsTable(nombre_tabla)
     62 print(f"¡Tabla '{nombre_tabla}' creada exitosamente con {df_estudiantes.count()} registros dummy!")

File <command-4801988587202543>, line 51
     49 # 4. Crear el DataFrame con los datos generados
     50 datos_dummy = generar_datos_estudiantes(1000)
---> 51 df_estudiantes = spark.createDataFrame(datos_dummy, schema=esquema)
     53 # 5. Guardar el DataFrame como una tabla Delta en Databricks
     54 # Cambia "default" por el nombre de tu 